In [1]:
import numpy as np
import pandas as pd
from tqdm import tqdm

from scipy.stats import pearsonr
from scipy.optimize import minimize
from scipy.optimize import differential_evolution
from scipy.interpolate import interp1d

from sklearn.metrics import r2_score

import seaborn as sns
import matplotlib.pyplot as plt

from numba import jit, float64, int64

In [2]:
@jit(nopython=True)
def simulate_CDDM(threshold, lamda, delta, loc, scale, ndt=0, z=0, sigma=1, dt=0.001):
    x = z
    
    rt = 0
    
    while -(threshold * np.exp(-lamda*rt)) < x and x < (threshold * np.exp(-lamda*rt)):
        x += delta * dt + sigma*np.sqrt(dt)*np.random.normal(0, 1)
        rt += dt
        
    if x >= (threshold * np.exp(-lamda*rt)):
        ch = 1
    else:
        ch = -1
        
    return (rt+ndt)*ch, np.random.lognormal(loc, scale)

In [3]:
@jit(nopython=True)
def series_fpt(t, v, a, w):
    s = np.zeros(t.shape)
    
    for k in range(1, 501):
        s += np.pi/a**2 * np.exp(-v*a*w - 0.5*v**2*t) * k * np.sin(k*np.pi*w) * np.exp(-0.5*k**2*np.pi**2*t/a**2)
    return s

In [4]:
@jit(nopython=True)
def f(x, t, z, tau, delta, sigma=1):
    term1 = 1/np.sqrt(2 * np.pi * sigma**2 * (t-tau))
    term2 = -(x - z - delta * (t-tau))**2 / (2 * sigma**2 * (t-tau))
    return term1 * np.exp(term2)

@jit(nopython=True)
def psi(threshold, lamda, t, z, tau, delta, sigma=1):
    db = -lamda * threshold * np.exp(-lamda*t)
    term1 = 0.5*f(threshold * np.exp(-lamda*t), t, z, tau, delta, sigma)
    term2 = db - delta - (threshold * np.exp(-lamda*t) - z - delta * (t-tau))/(t-tau)
    return term1 * term2

@jit(nopython=True)
def fpt(threshold, lamda, delta, z=0, sigma=1, dt=0.02, T_max=5):
    gu = np.zeros((int(T_max/dt)+2,))
    gl = np.zeros((int(T_max/dt)+2,))
    T = np.zeros((int(T_max/dt)+2,))
    
    gu[1] = -2*psi(threshold, lamda, dt, z, 0, delta, sigma)
    gl[1] =  2*psi(-threshold, lamda, dt, z, 0, delta, sigma)
    T[1] = dt
    
    for n in range(2, int(T_max/dt)+2):
        su = -2 * psi( threshold, lamda, n*dt, z, 0, delta, sigma)
        sl =  2 * psi(-threshold, lamda, n*dt, z, 0, delta, sigma)
        
        for j in range(1, n):
            if threshold * np.exp(-lamda*j*dt) == 0:
                continue
            
            psi_n_j_pp = psi( threshold, lamda, n*dt,  threshold * np.exp(-lamda*j*dt), j*dt, delta, sigma)
            psi_n_j_pn = psi( threshold, lamda, n*dt, -threshold * np.exp(-lamda*j*dt), j*dt, delta, sigma)
            psi_n_j_np = psi(-threshold, lamda, n*dt,  threshold * np.exp(-lamda*j*dt), j*dt, delta, sigma)
            psi_n_j_nn = psi(-threshold, lamda, n*dt, -threshold * np.exp(-lamda*j*dt), j*dt, delta, sigma)
            
            su +=  2 * dt * (gu[j] * psi_n_j_pp + gl[j] * psi_n_j_pn)
            sl += -2 * dt * (gu[j] * psi_n_j_np + gl[j] * psi_n_j_nn)
            
        gu[n] = su
        gl[n] = sl
        T[n] = (n*dt)
    return gu, gl, T

In [5]:
@jit(nopython=True)
def CDDM_likelihood_behv(prms, RT):
    delta = prms[2]
    sig = prms[3]
    t0 = prms[4]
    
    tt = np.maximum(np.abs(RT) - t0, 0)
    
    T_max = np.max(np.abs(RT))
    gu, gl, TT = fpt(prms[0], prms[1], delta, z=0, dt=0.05, T_max=T_max)

    gtup = np.interp(tt, TT, gu)
    gtlp = np.interp(tt, TT, gl)
    
    ll = 0
    for i in range(len(RT)):
        if np.abs(RT[i])-t0 > 0:
            if RT[i]>=0:                
                if gtup[i]>1e-14:
                    ll += -np.log(gtup[i])
                else:
                    ll += -np.log(1e-14)
            else:                
                if gtlp[i]>1e-14:
                    ll += -np.log(gtlp[i])
                else:
                    ll += -np.log(1e-14) 
        else:
            ll += -np.log(1e-14)
    
    return ll

@jit(nopython=True)
def DDM_likelihood_behv(prms, RT):
    v = prms[1]
    sig = prms[2]
    t0 = prms[3]
    eta = prms[4]
    
    tt = np.maximum(np.abs(RT) - t0, 0)
    fpt_z = series_fpt(tt, 0, 2*prms[0], 0.5) 
    
    ll = 0
    for i in range(len(RT)):
        if np.abs(RT[i])-t0 > 0:
            ratio = 1/np.sqrt(1 + eta**2 * (np.abs(RT[i])-t0))
            if RT[i]>=0:  
                power = -0.5*v**2/eta**2 + 0.5 * ((prms[0] * eta**2 + v)**2 / (eta**2 * (1 + eta**2 * (np.abs(RT[i])-t0))))
                density = ratio*np.exp(power)* fpt_z[i]
                
                if density>1e-14:
                    ll += -np.log(density)
                else:
                    ll += -np.log(1e-14)
            else:         
                power = -0.5*v**2/eta**2 + 0.5 * ((-prms[0] * eta**2 + v)**2 / (eta**2 * (1 + eta**2 * (np.abs(RT[i])-t0))))
                density = ratio*np.exp(power)* fpt_z[i]
                
                if density>1e-14:
                    ll += -np.log(density)
                else:
                    ll += -np.log(1e-14)  
        else:
            ll += -np.log(1e-14)
    
    return ll

In [6]:
@jit(nopython=True)
def CDDM_likelihood(prms, RT, Z):
    delta = prms[2]
    sig = prms[3]
    t0 = prms[4]
    
    tt = np.maximum(np.abs(RT) - t0, 0)
    
    T_max = np.max(np.abs(RT))
    gu, gl, TT = fpt(prms[0], prms[1], delta, z=0, dt=0.05, T_max=T_max)

    gtup = np.interp(tt, TT, gu)
    gtlp = np.interp(tt, TT, gl)
    
    ll = 0
    for i in range(len(RT)):
        if np.abs(RT[i])-t0 > 0:
            if RT[i]>=0:
                
                ll += 0.5*(np.log(Z[i]) - np.log(t0) + 0.5*sig**2)**2/sig**2 + 0.5*np.log(2*np.pi*sig**2*Z[i]**2)
                
                if gtup[i]>1e-14:
                    ll += -np.log(gtup[i])
                else:
                    ll += -np.log(1e-14)
            else:
                
                ll += 0.5*(np.log(Z[i]) - np.log(t0) + 0.5*sig**2)**2/sig**2 + 0.5*np.log(2*np.pi*sig**2*Z[i]**2)
                
                if gtlp[i]>1e-14:
                    ll += -np.log(gtlp[i])
                else:
                    ll += -np.log(1e-14) 
        else:
            ll += -np.log(1e-14)
    
    return ll

@jit(nopython=True)
def DDM_likelihood(prms, RT, Z):
    v = prms[1]
    sig = prms[2]
    t0 = prms[3]
    eta = prms[4]
    
    tt = np.maximum(np.abs(RT) - t0, 0)
    fpt_z = series_fpt(tt, 0, 2*prms[0], 0.5) 
    
    ll = 0
    for i in range(len(RT)):
        if np.abs(RT[i])-t0 > 0:
            ratio = 1/np.sqrt(1 + eta**2 * (np.abs(RT[i])-t0))
            if RT[i]>=0:
                
                ll += 0.5*(np.log(Z[i]) - np.log(t0) + 0.5*sig**2)**2/sig**2 + 0.5*np.log(2*np.pi*sig**2*Z[i]**2)
                
                power = -0.5*v**2/eta**2 + 0.5 * ((prms[0] * eta**2 + v)**2 / (eta**2 * (1 + eta**2 * (np.abs(RT[i])-t0))))
                density = ratio*np.exp(power)* fpt_z[i]
                
                if density>1e-14:
                    ll += -np.log(density)
                else:
                    ll += -np.log(1e-14)
            else:
                
                ll += 0.5*(np.log(Z[i]) - np.log(t0) + 0.5*sig**2)**2/sig**2 + 0.5*np.log(2*np.pi*sig**2*Z[i]**2)
                
                power = -0.5*v**2/eta**2 + 0.5 * ((-prms[0] * eta**2 + v)**2 / (eta**2 * (1 + eta**2 * (np.abs(RT[i])-t0))))
                density = ratio*np.exp(power)* fpt_z[i]
                
                if density>1e-14:
                    ll += -np.log(density)
                else:
                    ll += -np.log(1e-14)  
        else:
            ll += -np.log(1e-14)
    
    return ll

In [7]:
n_trials = 500
recovery_dic = {'True_model': [],
                'G2_ddm': [],
                'G2_ct_ddm': [],
                'BIC_ddm': [],
                'BIC_ct_ddm': [],
                'lambda': []}

In [8]:
for n in tqdm(range(100)):
    threshold = np.random.uniform(1.5, 4)
    if n%2 == 1:
        lamda = 0
        eta = np.random.uniform(0.05, .5)
    else:
        lamda = np.random.uniform(0.2, 2)
        eta = 0
    
    delta = np.random.uniform(0, 1)
    ndt = np.random.uniform(0.05, 1)
    scale_z = np.random.uniform(0.1, 1)
    loc_z = np.log(ndt) - 0.5 * scale_z**2
    
    RT = []
    Z = []
    
    for i in range(n_trials):
        delta_t = delta + eta*np.random.normal(0, 1)
        rt, z = simulate_CDDM(threshold, lamda, delta_t, loc_z, scale_z, ndt=ndt)
        RT.append(rt)
        Z.append(z)
        
    RT = np.array(RT)
    Z = np.array(Z)
    
    ans_cddm = differential_evolution(CDDM_likelihood,
                                      args=(RT, Z), 
                                      bounds=[(1.5, 4), (0, 2), (0, 3), 
                                              (0.05, 1), (0.05, 1)])
    
    ans_ddm = differential_evolution(DDM_likelihood,
                                      args=(RT, Z), 
                                      bounds=[(1.5, 4), (0, 3), 
                                              (0.05, 1), (0.05, 1), (0.05, 1)])
    
    if lamda == 0:
        recovery_dic['True_model'].append('DDM')
    else:
        recovery_dic['True_model'].append('CTDDM')
    
    recovery_dic['G2_ddm'].append(2*DDM_likelihood_behv(ans_ddm.x, RT))
    recovery_dic['G2_ct_ddm'].append(2*CDDM_likelihood_behv(ans_cddm.x, RT))
    recovery_dic['BIC_ddm'].append(2*DDM_likelihood_behv(ans_ddm.x, RT) + 4*np.log(n_trials))
    recovery_dic['BIC_ct_ddm'].append(2*CDDM_likelihood_behv(ans_cddm.x, RT) + 4*np.log(n_trials))
    recovery_dic['lambda'].append(ans_cddm.x[1])

100%|█████████████████████████████████████████| 100/100 [19:04<00:00, 11.45s/it]


In [9]:
recovery_df = pd.DataFrame(recovery_dic)
recovery_df['Best_model'] = 'CTDDM'
recovery_df.loc[recovery_df['BIC_ddm'] < recovery_df['BIC_ct_ddm'], 'Best_model'] = 'DDM'
recovery_df.loc[recovery_df['lambda'] < 0.1, 'Best_model'] = 'DDM'

In [10]:
file_name = '_data/Exponential.csv'.format(n_trials)
old_recovery_data = pd.read_csv(file_name, index_col=0)
recovery_df = pd.concat([old_recovery_data, 
                         recovery_df]).reset_index(drop=True)
recovery_df.to_csv(file_name)

In [11]:
(recovery_df['True_model'] == recovery_df['Best_model']).sum()/recovery_df.shape[0]

1.0

In [12]:
recovery_df

,True_model,G2_ddm,G2_ct_ddm,BIC_ddm,BIC_ct_ddm,lambda,Best_model
0,CTDDM,970.470086,586.783421,995.328519,611.641853,1.190070,CTDDM
1,DDM,3662.492975,3674.339673,3687.351407,3699.198106,0.000000,DDM
2,CTDDM,1119.937964,429.790450,1144.796397,454.648883,2.000000,CTDDM
3,DDM,1915.194388,1914.049927,1940.052821,1938.908359,0.012323,DDM
4,CTDDM,1034.415242,763.894384,1059.273674,788.752816,0.944640,CTDDM
...,...,...,...,...,...,...,...
295,DDM,2881.741253,2884.914694,2906.599685,2909.773126,0.000000,DDM
296,CTDDM,1430.288169,984.467998,1455.146602,1009.326431,1.166729,CTDDM
297,DDM,2979.968870,3026.520956,3004.827303,3051.379389,0.000000,DDM
298,CTDDM,970.242034,372.085525,995.100466,396.943957,1.842312,CTDDM
